In [1]:
# ① ライブラリ読み込み
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb

In [2]:
# ② データ読み込み
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")

print(train.shape)
print(test.shape)
display(train.head())

(630000, 21)
(270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [3]:
# ③ コピー作成
df_train = train.copy()
df_test = test.copy()

In [4]:
# ④ 目的変数・ID確認
target = "Irrigation_Need"
id_col = "id"

print(df_train.columns)
print(df_test.columns)
print(df_train[target].value_counts())

Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need'],
      dtype='object')
Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region'],
      dtype='object')
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


In [5]:
# ⑤ 特徴量エンジニアリング（毎日ここを追加）
features_new = []

# 例1：総水量
df_train["total_water"] = df_train["Rainfall_mm"] + df_train["Previous_Irrigation_mm"]
df_test["total_water"] = df_test["Rainfall_mm"] + df_test["Previous_Irrigation_mm"]
features_new.append("total_water")

# 例2：水分効率
df_train["water_efficiency"] = df_train["Soil_Moisture"] / (df_train["total_water"] + 1)
df_test["water_efficiency"] = df_test["Soil_Moisture"] / (df_test["total_water"] + 1)
features_new.append("water_efficiency")

# 例3：温度×湿度
df_train["temp_x_humidity"] = df_train["Temperature_C"] * df_train["Humidity"]
df_test["temp_x_humidity"] = df_test["Temperature_C"] * df_test["Humidity"]
features_new.append("temp_x_humidity")

print("追加特徴量:", features_new)
print(df_train.shape, df_test.shape)

追加特徴量: ['total_water', 'water_efficiency', 'temp_x_humidity']
(630000, 24) (270000, 23)


In [6]:
# ① 数値列ヒストグラム（一気に確認）
# ========================================
# 可視化①：ヒストグラム（数値全体）
# ========================================

# 何が分かる？
# 偏り（歪み）
# 外れ値
# 分布の形（正規っぽいか）

# import matplotlib.pyplot as plt

# num_cols = df_train.select_dtypes(include=["int64", "float64"]).columns

# df_train[num_cols].hist(figsize=(15, 10), bins=30)
# plt.suptitle("Numerical Features Distribution", fontsize=16)
# plt.show()

In [7]:
# ② ターゲット別ヒストグラム（超重要）
# ========================================
# 可視化②：ターゲット別分布
# ========================================
# ここが重要
# 分布がズレてる特徴量 = 強い特徴量候補

# for col in num_cols[:5]:  # 最初は5個だけでOK
#     plt.figure()
    
#     for label in sorted(df_train[target].unique()):
#         subset = df_train[df_train[target] == label]
#         plt.hist(subset[col], bins=30, alpha=0.5, label=f"class {label}")
    
#     plt.title(f"{col} by target")
#     plt.legend()
#     plt.show()

In [8]:
# ③ 相関ヒートマップ（全体把握）
# ========================================
# 可視化③：相関ヒートマップ
# ========================================
#  見るポイント
# 強い相関（0.8以上）
# ダブり特徴量（削除候補）

# import seaborn as sns

# plt.figure(figsize=(12, 10))
# corr = df_train[num_cols].corr()

# sns.heatmap(corr, cmap="coolwarm")
# plt.title("Correlation Heatmap")
# plt.show()

In [9]:
# ④ 1変数だけ深掘り（これ最強）
# ========================================
# 可視化④：単体チェック（重要）
# ========================================

# col = "Soil_Moisture"  # ←ここ変えるだけ

# plt.figure()
# for label in sorted(df_train[target].unique()):
#     subset = df_train[df_train[target] == label]
#     plt.hist(subset[col], bins=30, alpha=0.5, label=f"class {label}")

# plt.title(f"{col} distribution by target")
# plt.legend()
# plt.show()

In [10]:

#  使い方（超シンプル）

# ① ヒストグラム見る
# ② 「分布ズレてる列」探す
# ③ そこから特徴量作る

# あなた向けの見方（超重要）

# 例えば

# class0 → 低い
# class1 → 中間
# class2 → 高い

#  こうなってる列は
#  そのままでも強い or 掛け算でさらに強化できる

#  次の一手（ここから伸びる）

# このあと

#  「このグラフから何作る？」
#  一緒に“当たり特徴量”作れます

# スクショ出せば
#  ピンポイントで指示できます

In [11]:
# ⑥ X, y 作成
X = df_train.drop(columns=[target])
y = df_train[target].copy()

print(X.shape)
print(y.shape)

(630000, 23)
(630000,)


In [12]:
# ========================================
# ⑥.5 目的変数のラベルエンコード
# ========================================

from sklearn.preprocessing import LabelEncoder

target_le = LabelEncoder()
y = target_le.fit_transform(y)

print("target classes:", target_le.classes_)
print("encoded y sample:", y[:5])

target classes: ['High' 'Low' 'Medium']
encoded y sample: [1 1 1 2 1]


In [13]:
# ⑦ Label Encoding
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("カテゴリ列:", cat_cols)

for col in cat_cols:
    le = LabelEncoder()
    full_col = pd.concat([X[col], df_test[col]], axis=0).astype(str)
    le.fit(full_col)

    X[col] = le.transform(X[col].astype(str))
    df_test[col] = le.transform(df_test[col].astype(str))

print("Label Encoding 完了")

カテゴリ列: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Label Encoding 完了


In [14]:
# ⑧ 学習用test作成セルも少し修正
FEATURES = X.columns.tolist()
X_train = X.copy()
y_train = y.copy()
X_test = df_test[FEATURES].copy()

print(X_train.shape)
print(X_test.shape)
print(type(y_train))

(630000, 23)
(270000, 23)
<class 'numpy.ndarray'>


In [15]:
# =========================
# 多クラス分類 CV（AUC評価）差し替えセル
# Predicting Irrigation Need 用
# =========================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# -------------------------
# 目的変数
# -------------------------
target = "Irrigation_Need"

# 文字ラベル → 数値
target_map = {"Low": 0, "Medium": 1, "High": 2}
reverse_map = {0: "Low", 1: "Medium", 2: "High"}

df_train[target] = df_train[target].map(target_map)

# -------------------------
# 特徴量
# target, id を除外
# -------------------------
features = [col for col in df_train.columns if col not in [target, "id"]]

X_train = df_train[features].copy()
y_train = df_train[target].copy()
X_test = df_test[features].copy()

# object列があるとき用
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

# train/test の列を揃える
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)

# -------------------------
# クラス数
# -------------------------
num_classes = y_train.nunique()
print("num_classes:", num_classes)

# -------------------------
# LightGBM パラメータ
# -------------------------
params = {
    "boosting_type": "gbdt",
    "objective": "multiclass",
    "num_class": num_classes,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "n_estimators": 100000,
    "random_state": 42,
    "verbosity": -1,
}

# -------------------------
# CV
# -------------------------
n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

metrics = []
models = []
imp = pd.DataFrame()
test_preds = np.zeros((len(X_test), num_classes))

for nfold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n========== Fold {nfold + 1} / {n_splits} ==========")

    x_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    x_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        x_tr, y_tr,
        eval_set=[(x_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    # 確率予測
    y_tr_pred = model.predict_proba(x_tr)
    y_va_pred = model.predict_proba(x_va)

    # 多クラスAUC
    metric_tr = roc_auc_score(
        y_tr,
        y_tr_pred,
        multi_class="ovr",
        average="macro"
    )
    metric_va = roc_auc_score(
        y_va,
        y_va_pred,
        multi_class="ovr",
        average="macro"
    )

    print(f"[AUC] train: {metric_tr:.4f}, val: {metric_va:.4f}")

    metrics.append([nfold + 1, metric_tr, metric_va])
    models.append(model)

    test_preds += model.predict_proba(X_test) / n_splits

    _imp = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_,
        "fold": nfold + 1,
    })
    imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

# -------------------------
# CV結果まとめ
# -------------------------
metrics = np.array(metrics)

print("\n=== CV Result ===")
print(f"[CV Result] train: {metrics[:, 1].mean():.4f} ± {metrics[:, 1].std():.4f}")
print(f"[CV Result] val:   {metrics[:, 2].mean():.4f} ± {metrics[:, 2].std():.4f}")

X_train: (630000, 46)
y_train: (630000,)
X_test : (270000, 46)
num_classes: 3

========== Fold 1 / 5 ==========
[AUC] train: 0.9998, val: 0.9975

========== Fold 2 / 5 ==========
[AUC] train: 0.9998, val: 0.9975

========== Fold 3 / 5 ==========
[AUC] train: 0.9986, val: 0.9970

========== Fold 4 / 5 ==========
[AUC] train: 0.9998, val: 0.9977

========== Fold 5 / 5 ==========
[AUC] train: 0.9998, val: 0.9978

=== CV Result ===
[CV Result] train: 0.9996 ± 0.0005
[CV Result] val:   0.9975 ± 0.0002


In [16]:
# =========================
# 提出用セル
# =========================

pred_class = np.argmax(test_preds, axis=1)
pred_label = pd.Series(pred_class).map(reverse_map)

submission = pd.DataFrame({
    "id": df_test["id"],
    "Irrigation_Need": pred_label
})

submission.to_csv("submission.csv", index=False)
print("submission.csv 作成完了")
submission.head()

submission.csv 作成完了


,id,Irrigation_Need
0,630000,Low
1,630001,Medium
2,630002,Low
3,630003,Low
4,630004,Low


In [17]:
# 毎日の使い方

# 毎日触るのは基本ここだけです。

# 触るセル
# ⑤ 特徴量エンジニアリング
# 毎回実行する順番
# ⑤
# ⑥
# ⑦
# ⑧
# ⑨
# ⑩
# ⑪

In [18]:
# いちばん大事な注意
# 今後、特徴量を追加するときは必ず両方に入れます。
# df_train["新特徴量"] = ...
# df_test["新特徴量"] = ...
# これを片方だけにすると、あとで高確率でエラーになります。

In [19]:
# 最初の1個だけ試すなら
# まずは⑤をこの最小形にして回すのもアリです。
# これでCVを見て、上がるか確認です。
# features_new = []

# df_train["total_water"] = df_train["Rainfall_mm"] + df_train["Previous_Irrigation_mm"]
# df_test["total_water"] = df_test["Rainfall_mm"] + df_test["Previous_Irrigation_mm"]
# features_new.append("total_water")

# print("追加特徴量:", features_new)

In [20]:
# 次におすすめの特徴量3つ

# 次に足しやすいのはこの3つです。
# df_train["moisture_x_temp"] = df_train["Soil_Moisture"] * df_train["Temperature_C"]
# df_test["moisture_x_temp"] = df_test["Soil_Moisture"] * df_test["Temperature_C"]

# df_train["sun_per_temp"] = df_train["Sunlight_Hours"] / (df_train["Temperature_C"] + 1)
# df_test["sun_per_temp"] = df_test["Sunlight_Hours"] / (df_test["Temperature_C"] + 1)

# df_train["humidity_x_temp"] = df_train["Humidity"] * df_train["Temperature_C"]
# df_test["humidity_x_temp"] = df_test["Humidity"] * df_test["Temperature_C"]